# Notebook 05 Main · Documented-response draft

Run code Cell 1 on CPU using the current expanded Stage 03 and the corrected Stage 10 metadata-release dataset, each with its matching external manifest. Use one Stage 10 input. Cell 1 produces `/kaggle/working/stage05_main_v6_work/review/oracle_review_main_packets.zip`.

This draft keeps the original 850 operation IDs, applies the verified response-fault link repairs, and removes applicable scenarios without a documented response from the response-backed suite. The full excluded scenarios stay in `excluded_scenarios_audit.jsonl` for provenance and for qualifying how the evaluator will handle requests like those cases. Review `scenario_denominator_changes.csv` for the before/after counts. The old Stage 10 qualification was for the previous design. This draft clears neither Checkpoint B nor Main generation.


In [1]:
import csv
import hashlib
import io
import json
import os
import re
import zipfile
from concurrent.futures import ThreadPoolExecutor
from time import perf_counter
from pathlib import Path, PurePosixPath

INPUT = Path(os.environ.get("OPENAPI_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("OPENAPI_WORK", "/kaggle/working"))
DESIGN = "openapi-fresh-pilot-v1"


def sha(data):
    return hashlib.sha256(data).hexdigest()


def file_sha(path):
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()


def encode(data):
    return (json.dumps(data, sort_keys=True, ensure_ascii=False, separators=(",", ":"),
                       allow_nan=False) + "\n").encode()


def load(path):
    return json.loads(path.read_bytes())


def records(path):
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


def csv_records(path):
    with path.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


def atomic(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_bytes(data)
    tmp.replace(path)


def jsonl(path, rows):
    atomic(path, b"".join(map(encode, rows)))


def csv_write(path, fields, rows):
    output = io.StringIO(newline="")
    writer = csv.DictWriter(output, fieldnames=fields, lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)
    atomic(path, output.getvalue().encode())


def safe_path(name):
    p = PurePosixPath(name)
    if p.is_absolute() or ".." in p.parts or "\\" in name or p.as_posix() != name:
        raise ValueError(f"Unsafe manifest path: {name}")
    return Path(*p.parts)


import hashlib
import json
import re
from pathlib import PurePosixPath


ARTIFACT_SCHEMA = "openapi-artifact-v1"
PRODUCER_NOTEBOOKS = {
    "00": "00_protocol_and_runtime",
    "01": "01_corpus_preparation_and_eligibility",
    "02": "02_operation_slicing_and_prose_measurement",
    "03": "03_pair_and_evaluator_feasibility",
    "04": "04_strata_and_pilot_selection",
    "05": "05_oracle_build_and_review",
    "06": "06_build_corrected_evaluator_and_generation_tasks",
    "07": "07_pilot_generation",
    "08": "08_pilot_execution_and_review",
    "09": "09_pilot_scoring",
    "10": "10_power_and_main_sampling",
}


def provenance_digest(value):
    data = json.dumps(value, sort_keys=True, separators=(",", ":"),
                      ensure_ascii=False, allow_nan=False).encode("utf-8")
    return hashlib.sha256(data).hexdigest()


def artifact_path_for(record):
    paths = {record[key] for key in ("artifact_path", "relative_path", "path")
             if key in record}
    if len(paths) != 1:
        raise RuntimeError("Missing or inconsistent artifact paths")
    name = paths.pop()
    path = PurePosixPath(name)
    if not name or name == "." or path.is_absolute() or ".." in path.parts or "\\" in name or path.as_posix() != name:
        raise RuntimeError(f"Invalid artifact path: {name!r}")
    return name


def manifest_digest_valid(value):
    return isinstance(value, str) and re.fullmatch(r"[0-9a-f]{64}", value) is not None


def register_artifact_provenance(manifest):
    stage = manifest["stage"]
    artifacts = manifest["artifacts"]
    if "upstream_manifest_sha256" not in manifest:
        if stage != "00":
            raise RuntimeError(f"Stage {stage} has no upstream identity")
        manifest["upstream_manifest_sha256"] = {}
    links = manifest["upstream_manifest_sha256"]
    config_hash = manifest.get("configuration_hash")
    if config_hash is None:
        config_hash = provenance_digest({artifact_path_for(row): row["sha256"]
                                         for row in artifacts
                                         if artifact_path_for(row).startswith("config/")})
    manifest["configuration_hash"] = config_hash
    manifest["artifact_record_schema"] = ARTIFACT_SCHEMA
    upstream_hash = provenance_digest(links)
    for row in artifacts:
        row.update(artifact_path=artifact_path_for(row), schema_version=ARTIFACT_SCHEMA,
                   producer_notebook=PRODUCER_NOTEBOOKS[stage], config_hash=config_hash,
                   upstream_manifest_hash=upstream_hash)
    verify_artifact_provenance(manifest)


def verify_artifact_provenance(manifest):
    stage = manifest["stage"]
    if manifest.get("artifact_record_schema") != ARTIFACT_SCHEMA:
        raise RuntimeError(f"Stage {stage} requires a corrected provenance export")
    links = manifest["upstream_manifest_sha256"]
    config_hash = manifest["configuration_hash"]
    allowed = {key for key in PRODUCER_NOTEBOOKS if key < stage}
    if manifest.get("partition") == "main" and stage in ("05", "06"):
        allowed.add("10")
    if (not isinstance(links, dict) or bool(links) != (stage != "00")
            or any(key not in allowed or not manifest_digest_valid(value)
                   for key, value in links.items()) or not manifest_digest_valid(config_hash)):
        raise RuntimeError(f"Invalid Stage {stage} provenance")
    expected = (ARTIFACT_SCHEMA, PRODUCER_NOTEBOOKS[stage], config_hash, provenance_digest(links))
    seen = set()
    artifacts = manifest["artifacts"]
    if not artifacts or len(artifacts) != manifest["artifact_count"]:
        raise RuntimeError(f"Invalid Stage {stage} artifact count")
    for row in artifacts:
        name = artifact_path_for(row)
        actual = tuple(row.get(key) for key in
                       ("schema_version", "producer_notebook", "config_hash", "upstream_manifest_hash"))
        if (row.get("artifact_path") != name or name in seen or actual != expected
                or not manifest_digest_valid(row.get("sha256"))
                or type(row.get("bytes")) is not int or row["bytes"] < 0):
            raise RuntimeError(f"Invalid Stage {stage} artifact provenance: {name}")
        seen.add(name)


def mounted_release(stage, main=False, pin=None):
    suffix = "_main" if main and stage in {"05","06","07","08"} else ""
    found = []
    for sidecar in sorted(INPUT.rglob(f"stage_{stage}{suffix}_manifest.json")):
        external = load(sidecar)
        expected = external.get("internal_manifest_sha256")
        if external.get("stage") != stage or external.get("status") != "PASS":
            continue
        if main and external.get("partition") != "main":
            continue
        if pin and expected != pin:
            continue
        relative = f"registries/{stage}_manifest.json"
        candidates = [p for p in sidecar.parent.rglob(relative)
                      if p.is_file() and not p.is_symlink() and file_sha(p) == expected]
        if candidates:
            if len(candidates) != 1:
                raise RuntimeError(f"Ambiguous expanded Stage {stage} release.")
            found.append((sidecar, candidates[0], external))
    if len(found) != 1:
        raise RuntimeError(f"Attach one expanded, manifest-matched Stage {stage}{suffix} release. Found {len(found)}.")
    sidecar, registry, external = found[0]
    root = registry.parent.parent
    internal = load(registry)
    if (internal.get("stage") != stage or internal.get("status") != "PASS"
            or (main and internal.get("partition") != "main")):
        raise RuntimeError(f"Stage {stage}{suffix} status or partition differs.")
    verify_artifact_provenance(internal)
    artifacts = {}
    for row in internal["artifacts"]:
        relative = artifact_path_for(row)
        member = root / safe_path(relative)
        if (relative in artifacts or member.is_symlink() or not member.is_file()
                or member.stat().st_size != row["bytes"] or file_sha(member) != row["sha256"]):
            raise RuntimeError(f"Stage {stage}{suffix} invalid artifact {relative}.")
        artifacts[relative] = row["sha256"]
    return root, registry, sidecar, internal, external, artifacts


def stage_input(stage, *, main=False, pin=None):
    root, registry, _, internal, _, artifacts = mounted_release(stage, main, pin)
    return root, internal, file_sha(registry), artifacts


def pack(stage, root, metadata):
    output = WORK / f"stage_{stage}_main_dataset"
    output.mkdir(parents=True, exist_ok=True)
    manifest_path = root / f"registries/{stage}_manifest.json"
    files = sorted(p for p in root.rglob("*") if p.is_file() and p != manifest_path
                   and "__pycache__" not in p.parts and not p.name.endswith(".tmp"))
    artifacts = [{"relative_path": p.relative_to(root).as_posix(), "bytes": p.stat().st_size,
                  "sha256": file_sha(p)} for p in files]
    manifest = {"stage": stage, "status": "PASS", "design_id": DESIGN,
                "artifact_count": len(artifacts), "artifacts": artifacts, **metadata}
    register_artifact_provenance(manifest)
    atomic(manifest_path, encode(manifest))
    archive = output / f"stage_{stage}_main.zip"
    tmp = archive.with_suffix(".tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, compresslevel=4) as z:
        for p in [*files, manifest_path]:
            z.write(p, "prose_operation_experiment/" + p.relative_to(root).as_posix())
    with zipfile.ZipFile(tmp) as z:
        if z.testzip() is not None or len(z.namelist()) != len(artifacts) + 1:
            raise RuntimeError("Packaged Stage 05 archive integrity failed.")
        if any(sha(z.read("prose_operation_experiment/" + r["relative_path"])) != r["sha256"] for r in artifacts):
            raise RuntimeError("Packaged Stage 05 member hash differs.")
    tmp.replace(archive)
    external = {k: v for k, v in manifest.items() if k != "artifacts"}
    external.update(archive=archive.name, archive_sha256=file_sha(archive),
                    archive_bytes=archive.stat().st_size, internal_manifest_sha256=file_sha(manifest_path))
    atomic(output / f"stage_{stage}_main_manifest.json", encode(external))
    atomic(output / f"stage_{stage}_main_checksums.sha256",
           (f'{external["archive_sha256"]}  {archive.name}\n'
            f'{file_sha(output / f"stage_{stage}_main_manifest.json")}  stage_{stage}_main_manifest.json\n').encode())
    atomic(output / "README.md", f"# Stage {stage} main\n\nDesign {DESIGN}.\n".encode())
    print(f"Stage {stage} PASS. Publish {output}")
    return external



REVIEW_CHECKLIST = (
    "traceability", "baseline_validity", "variation_validity", "fault_validity",
    "scenario_mapping", "denominator_reconciliation", "ambiguities_resolved",
)


def approval_matches(row, expected_hash):
    return (
        row["packet_sha256"] == expected_hash
        and row["decision"] == "APPROVE"
        and all(row[field] == "PASS" for field in REVIEW_CHECKLIST)
        and bool(row["reviewer"].strip())
        and bool(row["evidence_notes"].strip())
    )


def qualify_packet(packet):
    document, package = packet["document"], packet["records"]
    oracle.validate_records(contracts, package)
    scenarios = {s["scenario_id"]: s for s in package["scenarios"]}
    baselines = {b["baseline_id"]: b for b in package["baselines"]}
    mapped = {key for key, s in scenarios.items() if s.get("documented_response_mapping") is not None}
    if Counter(b["scenario_id"] for b in baselines.values()) != Counter(mapped):
        raise RuntimeError(f"{packet['op_id']}: expected one baseline per mapped scenario.")
    for baseline in baselines.values():
        scenario = scenarios[baseline["scenario_id"]]
        oracle.audit_response(document, scenario, baseline, baseline)
        request = oracle.routing.prepare_frozen_fixture(document, scenario)
        match = oracle.routing.classify_request(request, package["scenarios"], document)
        if oracle.routing.select_frozen_response(match, package["baselines"]) != baseline:
            raise RuntimeError(f"{packet['op_id']}: baseline fixture replay changed scenario.")
    applicable = {r["requirement_id"] for r in package["requirements"] if r["applicability"] == "APPLICABLE"}
    for key, fault in (("legitimate_variations", False), ("response_faults", True)):
        for record in package[key]:
            baseline = baselines[record["baseline_id"]]
            scenario = scenarios[baseline["scenario_id"]]
            if (record.get("scenario_id") not in (None, baseline["scenario_id"])
                    or (fault and (record["requirement_id"] not in applicable
                                   or record["non_equivalent"] is not True))):
                raise RuntimeError(f"{packet['op_id']}: invalid replay record linkage.")
            oracle.audit_response(document, scenario, record, baseline, fault=fault)
    counts = {
        key: sum(r["applicability"] == "APPLICABLE" for r in package[key])
        for key in ("scenarios", "requirements", "response_faults")
    }
    if not all(counts.values()):
        raise RuntimeError(f"{packet['op_id']}: zero applicable scoring denominator.")
    return counts



import sys
import yaml
import jsonschema
from collections import Counter

if "date-time" not in jsonschema.FormatChecker().checkers:
    raise RuntimeError("Install rfc3339-validator==0.1.4 and restart before oracle qualification.")

stage10, manifest10, hash10, paths10 = stage_input("10", main=True)
stage03, manifest03, hash03, artifacts03 = stage_input(
    "03", pin=manifest10["upstream_manifest_sha256"]["03"])
if manifest10.get("checkpoint_d_reached") is not True or manifest10.get("may_begin_notebook_05_main") is not True:
    raise RuntimeError("A qualified Stage 10 Checkpoint D is required.")
import math


def matches_number(value, expected):
    return type(value) in (int, float) and math.isclose(value, expected, rel_tol=1e-12)


def check_stage10_qualification(plan, manifest):
    qualification = plan["qualification"]
    issues = []
    if manifest["operation_count"] != plan["size"]:
        issues.append(f"operation_count: manifest={manifest['operation_count']!r}, plan.size={plan['size']!r}")
    for field in ("fwer_qualified", "coverage_qualified", "power_goal_met"):
        value = qualification.get(field)
        if value is not True:
            issues.append(f"qualification.{field}: expected true, found {value!r}")
    acceptance = qualification.get("fwer_mc_acceptance", {})
    method = "one-sided exact lower-bound falsification check per null scenario"
    if acceptance.get("method") != method:
        issues.append(f"qualification.fwer_mc_acceptance.method: expected {method!r}, found {acceptance.get('method')!r}")
    nominal = acceptance.get("registered_alpha")
    if not matches_number(nominal, .05):
        issues.append(f"qualification.fwer_mc_acceptance.registered_alpha: expected 0.05, found {nominal!r}")
    confidence = acceptance.get("confidence")
    if type(confidence) not in (int, float) or not .90 <= confidence < 1:
        issues.append(f"qualification.fwer_mc_acceptance.confidence: expected [0.90, 1), found {confidence!r}")
    bound = qualification.get("highest_null_fwer_mc_lower")
    if type(bound) not in (int, float) or not 0 <= bound <= .05:
        issues.append(f"qualification.highest_null_fwer_mc_lower: expected [0, 0.05], found {bound!r}")
    if issues:
        raise RuntimeError("Stage 10 qualification: " + " | ".join(issues))
    return acceptance, bound


def check_analysis_metadata(plan, analysis, acceptance):
    issues = []
    labels = {
        "analysis_plan_version": "stage10-main-v4-power70-conservative-holm",
        "power_goal_authority": "SUPERVISOR_APPROVED_PROTOCOL_AMENDMENT",
    }
    for field, expected in labels.items():
        actual = analysis.get(field)
        if actual != expected:
            issues.append(f"main_analysis_registered.yaml::{field}: expected {expected!r}, found {actual!r}")
    numeric_fields = (
        ("main_analysis_registered.yaml::power_target", analysis.get("power_target"), .70),
        ("main_analysis_registered.yaml::nominal_familywise_alpha", analysis.get("nominal_familywise_alpha"), .05),
        ("main_analysis_registered.yaml::familywise_alpha", analysis.get("familywise_alpha"), .045),
        ("main_plan.yaml::analysis_decision_alpha", plan.get("analysis_decision_alpha"), .045),
        ("main_plan.yaml::qualification.fwer_mc_acceptance.analysis_decision_alpha",
         acceptance.get("analysis_decision_alpha"), .045),
    )
    for field, actual, expected in numeric_fields:
        if not matches_number(actual, expected):
            issues.append(f"{field}: expected {expected!r}, found {actual!r}")
    return issues


plan = yaml.safe_load((stage10 / "partitions/main/main_plan.yaml").read_bytes())
analysis = yaml.safe_load((stage10 / "partitions/main/main_analysis_registered.yaml").read_bytes())
fw, lower_bound = check_stage10_qualification(plan, manifest10)
STAGE10_ANALYSIS_ISSUES = check_analysis_metadata(plan, analysis, fw)
print(f"Verified Stage 10 checkpoint and simulation qualification for {plan['size']} Main pairs.")
print(f"Stage 10 internal manifest SHA-256: {hash10}; highest null FWER lower bound: {lower_bound:.6f}")
if STAGE10_ANALYSIS_ISSUES:
    print("Analysis metadata differs from the Main analysis contract. Draft review only; Cell 2 will hold publication.")
    for issue in STAGE10_ANALYSIS_ISSUES:
        print(" - " + issue)
else:
    print("Main analysis metadata matches the registered 0.045 decision alpha and 0.70 power target.")
selection = csv_records(stage10 / "partitions/main/selected.csv")
eligible = {r["op_id"]: r for r in csv_records(stage03 / "data/inventory/eligible_operations.csv")}
allocation, eligible_sizes = plan["n_h"], plan["N_h"]
if (len(selection) != plan["size"] or len({r["op_id"] for r in selection}) != len(selection)
        or len({r["pair_id"] for r in selection}) != len(selection)
        or Counter(r["stratum"] for r in selection) != allocation):
    raise RuntimeError("Main selection differs from Checkpoint D.")
for row in selection:
    authoritative = eligible[row["op_id"]]
    h = row["stratum"]
    if (row["final_status"] != "ELIGIBLE" or row["development_exposed"].lower() != "false"
            or any(authoritative[k] != row[k] for k in (
                "pair_id", "family_id", "spec_id", "method", "path", "L0_sha256",
                "L1_sha256", "oracle_sha256", "prose_tokens_L0"))
            or float(row["inclusion_probability"]) != allocation[h]/eligible_sizes[h]):
        raise RuntimeError("Main operation differs from the frozen eligible frame.")
    row.update(final_selection_status="RETAINED_MAIN_SELECTION", final_stratum_N=eligible_sizes[h],
               stratum_population_share=eligible_sizes[h]/sum(eligible_sizes.values()),
               inclusion_probability=float(row["inclusion_probability"]),
               sampling_weight=float(row["sampling_weight"]))
    for condition in ("L0","L1"):
        row[f"{condition}_hash"] = row[f"{condition}_sha256"]

qualification = load(stage03 / "reports/audit/feasibility_builder_qualification.json")
if any(artifacts03.get(f"src/{n}") != h for n,h in qualification["module_hashes"].items()):
    raise RuntimeError("Stage 03 oracle builder bytes differ.")
sys.path.insert(0, str(stage03 / "src"))
import oracle
if (Path(oracle.__file__).resolve() != (stage03 / "src/oracle.py").resolve()
        or oracle.ORACLE_VERSION != qualification["builder_version"]):
    raise RuntimeError("Restart with the frozen oracle builder.")
contracts_path = stage03 / "config/oracle_record_contracts.json"
contracts = load(contracts_path)
if artifacts03["config/oracle_record_contracts.json"] != qualification["contract_sha256"]:
    raise RuntimeError("Stage 03 oracle contracts differ.")

ORACLE_REPAIR_VERSION = "stage05-main-fault-context-link-v1"


def repair_fault_links(op_id, records):
    requirements = {item["requirement_id"]: item for item in records["requirements"]}
    if len(requirements) != len(records["requirements"]):
        raise RuntimeError(f"{op_id}: duplicate requirement IDs")
    repairs = []
    for fault in records["response_faults"]:
        validation = fault["validation_evidence"]
        linked = requirements[fault["requirement_id"]]
        rule = linked["expected_rule"]
        context = rule.get("context")
        if not isinstance(context, str) or not context.startswith("response_body:"):
            continue
        media_type = validation.get("media_type")
        if not media_type:
            continue
        expected = f"response_body:{validation['response_key']}:{media_type}"
        if context == expected:
            continue
        expected_rule = dict(rule, context=expected)
        candidates = [
            candidate for candidate in requirements.values()
            if candidate["applicability"] == "APPLICABLE"
            and candidate["source_pointer"] == linked["source_pointer"]
            and candidate["kind"] == linked["kind"]
            and candidate["expected_rule"] == expected_rule
        ]
        if len(candidates) != 1:
            raise RuntimeError(
                f"{op_id}/{fault['fault_id']}: expected one requirement in {expected}; "
                f"found {len(candidates)}"
            )
        fault["requirement_id"] = candidates[0]["requirement_id"]
        repairs.append({
            "op_id": op_id, "fault_id": fault["fault_id"], "baseline_id": fault["baseline_id"],
            "previous_requirement_id": linked["requirement_id"],
            "corrected_requirement_id": fault["requirement_id"],
            "previous_context": context, "expected_context": expected,
            "source_pointer": linked["source_pointer"],
        })
    return repairs


SCENARIO_POLICY = "main-documented-response-suite-v1"


def select_response_backed_scenarios(records):
    """Split the published opportunity suite from audited unsupported cases."""
    kept, excluded = [], []
    for scenario in records["scenarios"]:
        if (scenario["applicability"] == "APPLICABLE"
                and scenario["documented_response_mapping"] is None):
            excluded.append(scenario)
        else:
            kept.append(scenario)
    records["scenarios"] = kept
    return excluded


PROJECT = WORK / "stage05_main_v6_work/project"
REVIEW = WORK / "stage05_main_v6_work/review"
PACKETS = REVIEW / "packets"
COLUMNS = ["op_id", "packet_sha256", *REVIEW_CHECKLIST, "decision", "reviewer", "evidence_notes"]
review_rows = []
fault_repairs = []
unmapped_scenarios = []
scenario_counts = []
for row in selection:
    op = row["op_id"]
    l0 = stage03 / f"data/slices/L0/{op}.json"
    if file_sha(l0) != row["L0_hash"]:
        raise RuntimeError("Frozen L0 differs for " + op)
    document = load(l0)
    package = oracle.build_oracle(document, op, row["method"], row["path"], contracts)
    if sha(encode(package)) != row["oracle_sha256"]:
        raise RuntimeError("Main oracle differs from verified feasibility for " + op)
    upstream_oracle_sha256 = row["oracle_sha256"]
    repairs = repair_fault_links(op, package)
    excluded = select_response_backed_scenarios(package)
    oracle_sha256 = sha(encode(package))
    if repairs:
        for record in repairs:
            record.update(upstream_oracle_sha256=upstream_oracle_sha256, oracle_sha256=oracle_sha256)
        fault_repairs.extend(repairs)
    row["upstream_oracle_sha256"] = upstream_oracle_sha256
    row["oracle_sha256"] = oracle_sha256
    packet = {"op_id": op, "stage10_manifest_sha256": hash10, "l0_sha256": row["L0_hash"],
              "oracle_sha256": oracle_sha256, "document": document, "records": package}
    packet.update(upstream_oracle_sha256=upstream_oracle_sha256,
                  oracle_repair_version=ORACLE_REPAIR_VERSION,
                  scenario_policy=SCENARIO_POLICY)
    counts = qualify_packet(packet)
    scenario_counts.append({
        "op_id": op, "family_id": row["family_id"], "stratum": row["stratum"],
        "upstream_oracle_sha256": upstream_oracle_sha256, "draft_oracle_sha256": oracle_sha256,
        "original_applicable_scenarios": counts["scenarios"] + len(excluded),
        "response_backed_applicable_scenarios": counts["scenarios"],
        "excluded_applicable_scenarios": len(excluded),
        "applicable_requirements": counts["requirements"],
        "applicable_response_faults": counts["response_faults"],
    })
    for scenario in excluded:
        unmapped_scenarios.append({
            "op_id": op, "reason": "NO_DOCUMENTED_RESPONSE_MAPPING",
            "scenario": scenario,
        })
    packet_path = PACKETS / f"{op}.json"
    data = encode(packet)
    if packet_path.exists() and packet_path.read_bytes() != data:
        raise RuntimeError("Oracle packet changed for " + op)
    atomic(packet_path, data)
    review_rows.append({**dict.fromkeys(COLUMNS, ""), "op_id": op, "packet_sha256": sha(data)})
csv_write(REVIEW / "oracle_review_main_template.csv", COLUMNS, review_rows)
jsonl(REVIEW / "fault_link_repairs.jsonl", fault_repairs)
jsonl(REVIEW / "excluded_scenarios_audit.jsonl", unmapped_scenarios)
csv_write(REVIEW / "scenario_denominator_changes.csv", list(scenario_counts[0]), scenario_counts)
summary = {
    "operation_count": len(selection),
    "stage10_internal_manifest_sha256": hash10,
    "oracle_repair_version": ORACLE_REPAIR_VERSION,
    "repaired_fault_links": len(fault_repairs),
    "operations_with_repaired_fault_links": len({r["op_id"] for r in fault_repairs}),
    "scenario_policy": SCENARIO_POLICY,
    "checkpoint_b_reached": False,
    "main_generation_authorized": False,
    "stage10_requalification": "PENDING",
    "response_backed_applicable_scenarios": sum(r["response_backed_applicable_scenarios"] for r in scenario_counts),
    "excluded_applicable_scenarios": len(unmapped_scenarios),
    "operations_with_unmapped_scenarios": len({r["op_id"] for r in unmapped_scenarios}),
    "stage10_analysis_issues": STAGE10_ANALYSIS_ISSUES,
}
atomic(REVIEW / "main_oracle_draft_summary.json", encode(summary))
packet_archive = REVIEW / "oracle_review_main_packets.zip"
temporary_archive = packet_archive.with_suffix(".tmp")
with zipfile.ZipFile(temporary_archive, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in ("oracle_review_main_template.csv", "fault_link_repairs.jsonl",
                 "excluded_scenarios_audit.jsonl", "scenario_denominator_changes.csv",
                 "main_oracle_draft_summary.json"):
        archive.write(REVIEW / name, name)
    for row in review_rows:
        path = PACKETS / (row["op_id"] + ".json")
        archive.write(path, "packets/" + path.name)
with zipfile.ZipFile(temporary_archive) as archive:
    if archive.testzip() is not None or len(archive.namelist()) != len(selection) + 5:
        raise RuntimeError("Draft archive integrity check failed.")
    for row in review_rows:
        if sha(archive.read(f"packets/{row['op_id']}.json")) != row["packet_sha256"]:
            raise RuntimeError("Draft packet hash check failed for " + row["op_id"])
temporary_archive.replace(packet_archive)
print(json.dumps(summary, indent=2))
print(packet_archive)
print("DRAFT ONLY: Stage 10 requalification and the downstream evaluator check are pending.")


Verified Stage 10 checkpoint and simulation qualification for 850 Main pairs.
Stage 10 internal manifest SHA-256: 7978d8f18a1ceabeeb6454a87d310d4b94867572317d2dc1f5d8394a7893c17e; highest null FWER lower bound: 0.041431
Main analysis metadata matches the registered 0.045 decision alpha and 0.70 power target.
{
  "operation_count": 850,
  "stage10_internal_manifest_sha256": "7978d8f18a1ceabeeb6454a87d310d4b94867572317d2dc1f5d8394a7893c17e",
  "oracle_repair_version": "stage05-main-fault-context-link-v1",
  "repaired_fault_links": 423,
  "operations_with_repaired_fault_links": 71,
  "scenario_policy": "main-documented-response-suite-v1",
  "checkpoint_b_reached": false,
  "main_generation_authorized": false,
  "stage10_requalification": "PENDING",
  "response_backed_applicable_scenarios": 974,
  "excluded_applicable_scenarios": 8728,
  "operations_with_unmapped_scenarios": 809,
  "stage10_analysis_issues": []
}
/kaggle/working/stage05_main_v6_work/review/oracle_review_main_packets.zip
DR

## Publication pending

The publication cell belongs in the version that consumes the requalified plan and the final review decisions. This draft supplies the evidence for that qualification.
